# Figure 3 — Simulation Performance

Three-panel figure: (a) parameter adherence c-stat, (b) clinical fidelity, (c) realism (Turing test).

In [ ]:
import json
from math import sqrt
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.stats import gaussian_kde

ROOT = Path("..")
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

# ── Load data ──
with open(ROOT / "eval/report/data/comparative_metrics.json") as f:
    adherence = json.load(f)

with open(ROOT / "eval/report/data/fidelity_metrics.json") as f:
    fidelity = json.load(f)

with open(ROOT / "eval/turing/report/data/llm_sequential_metrics.json") as f:
    llm_turing = json.load(f)

with open(ROOT / "eval/turing/report/data/human_metrics.json") as f:
    human_turing = json.load(f)

In [ ]:
# ── Helpers ──

def wilson_ci(k: int, n: int) -> tuple[float, float]:
    z = 1.96
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    margin = z * sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(centre - margin, 0.0), min(centre + margin, 1.0)


# Group ordering and colours
GROUP_ORDER = ["communication_params", "persona_params",
               "content_params", "strategy_params"]
GROUP_COLORS = {
    "communication_params": "#3498db",
    "persona_params": "#e74c3c",
    "content_params": "#57bb8a",
    "strategy_params": "#9b59b6",
}
GROUP_LABELS = {
    "communication_params": "Communication",
    "persona_params": "Feelings",
    "content_params": "Content",
    "strategy_params": "Strategy",
}

In [ ]:
# ── Panel (a): Parameter adherence ──

params = adherence["per_param"]
rows = []
for key, val in params.items():
    lo, hi = wilson_ci(val["n_correct"], val["n_pairs"])
    rows.append({
        "label": val["label"],
        "cstat": val["auc"],
        "ci_lo": lo,
        "ci_hi": hi,
        "group": val["group_id"],
        "n": val["n_pairs"],
    })

# Sort by group order, then by c-stat within group
rows.sort(key=lambda r: (GROUP_ORDER.index(r["group"]), r["cstat"]))
macro_cstat = adherence["macro_auc"]

In [ ]:
# ── Panel (b): Clinical fidelity — per-vignette distributions ──

per_vig = fidelity["per_vignette"]
ov = fidelity["overall"]

per_vig_acc = np.array([v["mean_accuracy"] for v in per_vig.values()])
per_vig_plaus = np.array([v["mean_plausibility"]
                          for v in per_vig.values()
                          if v["n_unsupported"] > 0])

n_supported = sum(v["n_supported"] for v in per_vig.values())
n_unsupported = sum(v["n_unsupported"] for v in per_vig.values())

In [ ]:
# ── Panel (c): Turing test ──

claude_llm = next(r for r in llm_turing if r["label"] == "claude-opus-4.6")
gpt_llm = next(r for r in llm_turing if r["label"] == "gpt-5.4")
human = human_turing

def build_rater(r):
    n_total = r["tp"] + r["fn"] + r["fp"] + r["tn"]
    det_lo, det_hi = wilson_ci(r["tp"], r["tp"] + r["fn"])
    spec_lo, spec_hi = wilson_ci(r["tn"], r["tn"] + r["fp"])
    acc_lo, acc_hi = wilson_ci(r["tp"] + r["tn"], n_total)
    return {
        "label": r["label"],
        "det_rate": r["det_rate"], "det_lo": det_lo, "det_hi": det_hi,
        "specificity": r.get("specificity", r.get("real_recog_rate")),
        "spec_lo": spec_lo, "spec_hi": spec_hi,
        "accuracy": r["accuracy"], "acc_lo": acc_lo, "acc_hi": acc_hi,
    }

raters = [build_rater(gpt_llm), build_rater(claude_llm), build_rater(human)]

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURE 3
# ══════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 7.0))

TITLE_Y = 0.97

# Outer grid: 3 columns — lowered top to leave room for fig.text titles
outer = gridspec.GridSpec(1, 3, width_ratios=[1.1, 0.8, 1.0],
                          wspace=0.30, left=0.05, right=0.97,
                          top=0.82, bottom=0.16)

# Panel (b) gets an inner 2-row grid for the two KDE subplots
inner_b = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[1],
                                           hspace=0.60)

# ── Panel titles — vertically aligned across all three panels ─
a_bbox = outer[0].get_position(fig)
b_bbox = outer[1].get_position(fig)
c_bbox = outer[2].get_position(fig)

fig.text(a_bbox.x0, TITLE_Y,
         "(a) Parameter Adherence,\n     Claude Opus 4.6 as a Judge",
         fontsize=12, va="top", ha="left")
fig.text(b_bbox.x0, TITLE_Y,
         "(b) Clinical Fidelity,\n     Claude Opus 4.6 as a Judge",
         fontsize=12, va="top", ha="left")
fig.text(c_bbox.x0, TITLE_Y,
         "(c) Realism Evaluation,\n     Turing-inspired Test",
         fontsize=12, va="top", ha="left")

# ── (a) Parameter adherence ──────────────────────────────────
ax_a = fig.add_subplot(outer[0])

y_positions = []
current_y = 0
prev_group = None
for r in rows:
    if prev_group is not None and r["group"] != prev_group:
        current_y += 0.5
    y_positions.append(current_y)
    current_y += 1
    prev_group = r["group"]
y_pos = np.array(y_positions)

bar_colors = [GROUP_COLORS[r["group"]] for r in rows]
cstats = [r["cstat"] for r in rows]

ax_a.barh(y_pos, cstats, color=bar_colors, edgecolor="none",
          height=0.7, zorder=3)

for yp, cs in zip(y_pos, cstats):
    ax_a.text(cs + 0.008, yp, f"{cs:.2f}",
              va="center", ha="left", fontsize=7.5, color="0.3")

ax_a.axvline(0.5, color="grey", linestyle="--", linewidth=0.8, zorder=2)
ax_a.axvline(macro_cstat, color="black", linestyle="-", linewidth=1.2,
             zorder=2)
ax_a.text(macro_cstat + 0.005, y_pos[-1] + 0.8,
          f"Macro c-stat = {macro_cstat:.2f}",
          fontsize=8, color="black", va="bottom", ha="left")

ax_a.set_yticks(y_pos)
ax_a.set_yticklabels([r["label"] for r in rows], fontsize=9)
ax_a.set_xlim(0.4, 1.02)
ax_a.set_xlabel("Concordance statistic (c-stat)", fontsize=10)
ax_a.spines[["top", "right"]].set_visible(False)
ax_a.grid(axis="x", alpha=0.2, zorder=0)

from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=GROUP_COLORS[g], label=GROUP_LABELS[g])
                  for g in GROUP_ORDER]
ax_a.legend(handles=legend_patches, fontsize=8,
            loc="upper center", bbox_to_anchor=(0.5, -0.12),
            ncol=4, framealpha=0.9, title="Parameter Type",
            title_fontproperties={"size": 8.5})


# ── (b) Clinical fidelity — two KDE plots ────────────────────
kde_color = "#3498db"
mean_color = "#e74c3c"

# Top: Accuracy (supported claims)
ax_b1 = fig.add_subplot(inner_b[0])
x_acc = np.linspace(3.0, 4.0, 300)
kde_acc = gaussian_kde(per_vig_acc, bw_method=0.3)
ax_b1.fill_between(x_acc, kde_acc(x_acc), alpha=0.35, color=kde_color, zorder=3)
ax_b1.plot(x_acc, kde_acc(x_acc), color=kde_color, linewidth=1.5, zorder=4)
ax_b1.axvline(ov["mean_accuracy"], color=mean_color, linestyle="-",
              linewidth=1.5, zorder=5)
peak_acc = kde_acc(x_acc).max()
ax_b1.text(ov["mean_accuracy"] - 0.01, peak_acc * 0.85,
           f"Mean = {ov['mean_accuracy']:.2f}",
           fontsize=8.5, color=mean_color, ha="right", va="bottom")
ax_b1.scatter(per_vig_acc, np.zeros_like(per_vig_acc) - 0.05,
              marker="|", s=20, color=kde_color, alpha=0.6, zorder=3)
ax_b1.set_xlim(3.0, 4.0)
ax_b1.set_ylabel("Density", fontsize=9)
ax_b1.set_xlabel("Accuracy (1–4)", fontsize=9)
ax_b1.spines[["top", "right"]].set_visible(False)
ax_b1.set_yticks([])
ax_b1.set_title(f"Supported claims (N={n_supported:,})",
                fontsize=10, loc="left", pad=10)

# Bottom: Plausibility (unsupported claims)
ax_b2 = fig.add_subplot(inner_b[1])
x_plaus = np.linspace(3.0, 4.0, 300)
kde_plaus = gaussian_kde(per_vig_plaus, bw_method=0.3)
ax_b2.fill_between(x_plaus, kde_plaus(x_plaus), alpha=0.35, color="#57bb8a", zorder=3)
ax_b2.plot(x_plaus, kde_plaus(x_plaus), color="#57bb8a", linewidth=1.5, zorder=4)
ax_b2.axvline(ov["mean_plausibility"], color=mean_color, linestyle="-",
              linewidth=1.5, zorder=5)
peak_plaus = kde_plaus(x_plaus).max()
ax_b2.text(ov["mean_plausibility"] - 0.01, peak_plaus * 0.85,
           f"Mean = {ov['mean_plausibility']:.2f}",
           fontsize=8.5, color=mean_color, ha="right", va="bottom")
ax_b2.scatter(per_vig_plaus, np.zeros_like(per_vig_plaus) - 0.03,
              marker="|", s=20, color="#57bb8a", alpha=0.6, zorder=3)
ax_b2.set_xlim(3.0, 4.0)
ax_b2.set_ylabel("Density", fontsize=9)
ax_b2.set_xlabel("Plausibility (1–4)", fontsize=9)
ax_b2.spines[["top", "right"]].set_visible(False)
ax_b2.set_yticks([])
ax_b2.set_title(f"Unsupported claims (N={n_unsupported:,})",
                fontsize=10, loc="left", pad=10)


# ── (c) Turing test — grouped bars ───────────────────────────
ax_c = fig.add_subplot(outer[2])

metrics_c = ["det_rate", "specificity", "accuracy"]
metric_labels = ["Simulated\nDetection", "Real\nRecognition", "Overall\nAccuracy"]
rater_display = ["GPT-5.4", "Claude Opus 4.6", "Human"]
rater_colors = ["#1a5276", "#2e86c1", "#e74c3c"]

x = np.arange(len(metrics_c))
n_raters = len(raters)
bw = 0.25

for i, (rater, color, disp) in enumerate(zip(raters, rater_colors, rater_display)):
    vals = [rater[m] for m in metrics_c]
    lo_keys = ["det_lo", "spec_lo", "acc_lo"]
    hi_keys = ["det_hi", "spec_hi", "acc_hi"]
    lo = [rater[m] - rater[lk] for m, lk in zip(metrics_c, lo_keys)]
    hi = [rater[hk] - rater[m] for m, hk in zip(metrics_c, hi_keys)]
    offset = (i - (n_raters - 1) / 2) * bw
    bars = ax_c.bar(x + offset, vals, bw, color=color, edgecolor="none",
                    label=disp, zorder=3)
    ax_c.errorbar(x + offset, vals, yerr=[lo, hi],
                  fmt="none", ecolor="black", elinewidth=0.8,
                  capsize=2, zorder=4)
    for j, (bar, v) in enumerate(zip(bars, vals)):
        err_top = v + hi[j]
        ax_c.text(bar.get_x() + bar.get_width() / 2, err_top + 0.02,
                  f"{v:.0%}", ha="center", va="bottom", fontsize=7, color="0.3")

ax_c.axhline(0.5, color="grey", linestyle="--", linewidth=0.8, zorder=2)
ax_c.text(x[-1] + 0.45, 0.5, "Chance", va="center", ha="left",
          fontsize=8, color="grey")

ax_c.set_xticks(x)
ax_c.set_xticklabels(metric_labels, fontsize=10)
ax_c.set_ylim(0, 1.12)
ax_c.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax_c.set_ylabel("Rate", fontsize=10)
ax_c.spines[["top", "right"]].set_visible(False)
ax_c.grid(axis="y", alpha=0.2, zorder=0)
ax_c.legend(fontsize=9, loc="upper center",
            bbox_to_anchor=(0.5, -0.14), ncol=3,
            framealpha=0.9, title="Grader",
            title_fontproperties={"size": 9})


# ── Save ─────────────────────────────────────────────────────
fig.savefig(FIG_DIR / "figure3.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "figure_3.pdf", bbox_inches="tight")
plt.show()
print(f"Saved to {FIG_DIR / 'figure3.png'} and .pdf")